# FRED Gold Aggregation
Builds a Dim Date table, a long-format fact table, and pre-aggregated analytics tables for the Power BI dashboard.

### Step 1: Read Silver and build Dim Date table

In [0]:
import pandas as pd

# Read Silver
df_gold_source = spark.table("silver_fred_economic").toPandas()

# Build Dim Date table spanning the full range of our data
min_date = df_gold_source['date'].min()
max_date = df_gold_source['date'].max()

dim_date = pd.DataFrame({'date': pd.date_range(start=min_date, end=max_date, freq='D')})
dim_date['year'] = dim_date['date'].dt.year
dim_date['month_number'] = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.strftime('%B')
dim_date['quarter'] = 'Q' + dim_date['date'].dt.quarter.astype(str)
dim_date['year_month'] = dim_date['date'].dt.strftime('%Y-%m')
dim_date['day_of_week'] = dim_date['date'].dt.strftime('%A')

print(f"Dim Date records: {len(dim_date)}")
print(f"Date range: {min_date.date()} to {max_date.date()}")
display(dim_date.head(5))

Dim Date records: 29389
Date range: 1946-01-01 to 2026-06-18


date,year,month_number,month_name,quarter,year_month,day_of_week
1946-01-01T00:00:00.000Z,1946,1,January,Q1,1946-01,Tuesday
1946-01-02T00:00:00.000Z,1946,1,January,Q1,1946-01,Wednesday
1946-01-03T00:00:00.000Z,1946,1,January,Q1,1946-01,Thursday
1946-01-04T00:00:00.000Z,1946,1,January,Q1,1946-01,Friday
1946-01-05T00:00:00.000Z,1946,1,January,Q1,1946-01,Saturday


### Step 2: Build fact table and pre-aggregated analytics tables

In [0]:
# Fact table - already in long format from Silver, just rename for clarity
fact_economic = df_gold_source[['date', 'series_id', 'series_name', 'value']].copy()

# Latest value per series (for KPI cards)
latest_values = (
    fact_economic.sort_values('date')
    .groupby('series_id')
    .tail(1)
    .reset_index(drop=True)
)

# Year-over-year change table
fact_economic['year'] = fact_economic['date'].dt.year
yearly_avg = fact_economic.groupby(['series_id', 'series_name', 'year'])['value'].mean().reset_index()
yearly_avg = yearly_avg.sort_values(['series_id', 'year'])
yearly_avg['prior_year_value'] = yearly_avg.groupby('series_id')['value'].shift(1)
yearly_avg['yoy_change_pct'] = ((yearly_avg['value'] - yearly_avg['prior_year_value']) / yearly_avg['prior_year_value']) * 100

print("Latest values per series:")
display(latest_values)
print(f"\nYearly average records: {len(yearly_avg)}")
display(yearly_avg.tail(10))

Latest values per series:


date,series_id,series_name,value
2026-01-01T00:00:00.000Z,GDP,Gross Domestic Product,31819.464
2026-05-01T00:00:00.000Z,FEDFUNDS,Federal Funds Rate,3.63
2026-05-01T00:00:00.000Z,UNRATE,Unemployment Rate,4.3
2026-05-01T00:00:00.000Z,CPIAUCSL,Consumer Price Index,333.979
2026-06-18T00:00:00.000Z,MORTGAGE30US,30-Year Mortgage Rate,6.47



Yearly average records: 369


series_id,series_name,year,value,prior_year_value,yoy_change_pct
UNRATE,Unemployment Rate,2017,4.358333333333333,4.875,-10.598290598290596
UNRATE,Unemployment Rate,2018,3.891666666666666,4.358333333333333,-10.707456978967508
UNRATE,Unemployment Rate,2019,3.6750000000000003,3.891666666666666,-5.56745182012846
UNRATE,Unemployment Rate,2020,8.1,3.6750000000000003,120.40816326530607
UNRATE,Unemployment Rate,2021,5.3500000000000005,8.1,-33.950617283950606
UNRATE,Unemployment Rate,2022,3.65,5.3500000000000005,-31.775700934579447
UNRATE,Unemployment Rate,2023,3.625,3.65,-0.6849315068493127
UNRATE,Unemployment Rate,2024,4.0249999999999995,3.625,11.034482758620676
UNRATE,Unemployment Rate,2025,4.263636363636364,4.0249999999999995,5.928853754940726
UNRATE,Unemployment Rate,2026,4.32,4.263636363636364,1.3219616204690892


### Step 3: Write Gold tables to Delta

In [0]:
# Write Dim Date
spark.createDataFrame(dim_date).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("gold_dim_date")

# Write Fact table
spark.createDataFrame(fact_economic).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("gold_fact_economic")

# Write Latest Values
spark.createDataFrame(latest_values).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("gold_latest_values")

# Write Yearly YoY Change
spark.createDataFrame(yearly_avg).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("gold_yearly_yoy_change")

print("All Gold tables written successfully:")
print("- gold_dim_date")
print("- gold_fact_economic")
print("- gold_latest_values")
print("- gold_yearly_yoy_change")

All Gold tables written successfully:
- gold_dim_date
- gold_fact_economic
- gold_latest_values
- gold_yearly_yoy_change
